In [1]:
import tz_pypsa
import pandas as pd
import numpy as np
from tz_pypsa.model import Model
from google.cloud import bigquery

In [2]:
# Extract actual data from data warehouse
client = bigquery.Client()

query = """
    SELECT *
    FROM `tz-data-dev.taiwan_lnd.power_breakdown`
"""
query_job = client.query(query)

# Convert the result to a pandas DataFrame
df = query_job.to_dataframe()

In [4]:
df['datetime_local'] = df['datetime'].dt.tz_convert('Asia/Taipei')

In [13]:
gen_act = df.copy()
gen_act.sort_values(by='datetime', inplace=True, ignore_index=True)

# Select relevant columns and rename them
gen_act = gen_act[['datetime',
                   'datetime_local',
                   'powerConsumptionTotal',
                   'production_nuclear',
                   'production_geothermal', 
                   'production_biomass', 
                   'production_coal',
                   'production_wind', 
                   'production_solar', 
                   'production_hydro',
                   'production_gas', 
                   'production_oil', 
                   'production_unknown',
                   'production_hydro discharge',
]]

# Rename columns to match the model's output
gen_act.rename(columns={
    'powerConsumptionTotal': 'Demand',
    'production_nuclear': 'Nuclear',
    'production_geothermal': 'Geothermal',
    'production_biomass': 'Biomass',
    'production_coal': 'Coal',
    'production_wind': 'Wind',
    'production_solar': 'Solar',
    'production_hydro': 'Hydro',
    'production_gas': 'LNG',
    'production_oil': 'Oil',
    'production_unknown': 'CoGen',
    'production_hydro discharge': 'PumpedHydro',
}, inplace=True)

In [15]:
generation = pd.melt(
    gen_act,
    id_vars=['datetime', 'datetime_local'],
    var_name='Tech',
    value_name='Value'
)

In [18]:
generation.to_csv('historical_generation.csv', index=False)